# Credit Card Fraud Detection — TrustGuard

**Goal:** Build a high-recall, low-false-positive fraud detection system on highly imbalanced transaction data.

**Dataset:** [Credit Card Fraud Detection (Kaggle)](https://www.kaggle.com/datasets/mlg-ulb/brml/creditcardfraud) — 284,807 transactions, 492 frauds (0.17%)

**Pipeline:**
1. EDA + class imbalance analysis
2. Preprocessing (scaling, train/test split with stratification)
3. Baseline model (Logistic Regression)
4. Handle imbalance (SMOTE / class weights)
5. Train XGBoost / LightGBM
6. Evaluate: Precision, Recall, F1, AUC-ROC, AUC-PR, confusion matrix
7. Feature importance (SHAP)
8. Threshold tuning for business tradeoff (precision vs recall)

## 0. Setup

In [ ]:
!pip install -q imbalanced-learn xgboost lightgbm shap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, roc_curve, auc, average_precision_score, f1_score
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load Data

Download `creditcard.csv` from Kaggle and upload to Colab, or use the Kaggle API:
```python
from google.colab import files
files.upload()  # upload kaggle.json
!mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
!kaggle datasets download -d mlg-ulb/creditcardfraud --unzip
```

In [ ]:
df = pd.read_csv('creditcard.csv')
print(df.shape)
df.head()

## 2. EDA — Class Imbalance

In [ ]:
fraud_count = df['Class'].value_counts()
fraud_pct = df['Class'].value_counts(normalize=True) * 100
print(fraud_count)
print(fraud_pct)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(x='Class', data=df, ax=ax[0])
ax[0].set_title('Class Distribution (raw counts)')
ax[0].set_yscale('log')

sns.histplot(df['Amount'], bins=50, ax=ax[1])
ax[1].set_title('Transaction Amount Distribution')
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
X = df.drop(columns=['Class'])
y = df['Class']

scaler = StandardScaler()
X[['Time', 'Amount']] = scaler.fit_transform(X[['Time', 'Amount']])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train fraud rate: {y_train.mean():.4%}")
print(f"Test fraud rate:  {y_test.mean():.4%}")

## 4. Baseline — Logistic Regression (class-weighted)

In [ ]:
baseline = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
baseline.fit(X_train, y_train)

y_pred_base = baseline.predict(X_test)
y_proba_base = baseline.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred_base, digits=4))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba_base):.4f}")
print(f"AUC-PR:  {average_precision_score(y_test, y_proba_base):.4f}")

## 5. SMOTE Resampling + XGBoost

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE: {y_train.value_counts().to_dict()}")
print(f"After SMOTE:  {pd.Series(y_train_sm).value_counts().to_dict()}")

xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    eval_metric='aucpr',
    random_state=RANDOM_STATE,
    n_jobs=-1
)
xgb_model.fit(X_train_sm, y_train_sm)

## 6. Evaluation

In [ ]:
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=4))
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.4f}")
print(f"AUC-PR:  {average_precision_score(y_test, y_proba):.4f}")
print(f"F1:      {f1_score(y_test, y_pred):.4f}")

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'], yticklabels=['Legit', 'Fraud'])
plt.title('Confusion Matrix — XGBoost (SMOTE)')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
# ROC and Precision-Recall curves
fpr, tpr, _ = roc_curve(y_test, y_proba)
precision, recall, _ = precision_recall_curve(y_test, y_proba)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

ax[0].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, y_proba):.4f}')
ax[0].plot([0, 1], [0, 1], 'k--')
ax[0].set_xlabel('False Positive Rate')
ax[0].set_ylabel('True Positive Rate')
ax[0].set_title('ROC Curve')
ax[0].legend()

ax[1].plot(recall, precision, label=f'AUC-PR = {average_precision_score(y_test, y_proba):.4f}')
ax[1].set_xlabel('Recall')
ax[1].set_ylabel('Precision')
ax[1].set_title('Precision-Recall Curve')
ax[1].legend()

plt.tight_layout()
plt.show()

## 7. Feature Importance (SHAP)

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
# Sample for speed
X_sample = X_test.sample(2000, random_state=RANDOM_STATE)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values, X_sample, max_display=15)

## 8. Threshold Tuning — Business Tradeoff

Default threshold (0.5) isn't always optimal. For fraud detection, missing a fraud (false negative)
is usually costlier than a false alarm (false positive). Sweep thresholds to find the best tradeoff.

In [ ]:
thresholds = np.arange(0.1, 0.95, 0.05)
results = []

for t in thresholds:
    preds = (y_proba >= t).astype(int)
    results.append({
        'threshold': round(t, 2),
        'precision': classification_report(y_test, preds, output_dict=True)['1']['precision'],
        'recall': classification_report(y_test, preds, output_dict=True)['1']['recall'],
        'f1': f1_score(y_test, preds)
    })

results_df = pd.DataFrame(results)
results_df

## 9. Save Model

In [ ]:
import joblib
joblib.dump(xgb_model, 'fraud_xgb_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print("Model and scaler saved.")

## 10. Summary of Results

Fill in your final numbers here after running the notebook:

| Model | Precision | Recall | F1 | AUC-ROC | AUC-PR |
|---|---|---|---|---|---|
| Logistic Regression (baseline) | | | | | |
| XGBoost + SMOTE | | | | | |